In [15]:
import pandas as pd
import numpy as np
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import f1_score, make_scorer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.base import clone
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report
from imblearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

In [33]:
# Import the functions from the prep_functions notebook
%run "./prep_functions.ipynb"

# Check if the main functions and variables are imported correctly
print("Function imported correctly:")
print(f"- preprocess_train_df: {preprocess_train_df.__name__}")
print(f"- preprocess_test_df: {preprocess_test_df.__name__}")
print(f"- split_sets: {split_sets.__name__}")

Function imported correctly:
- preprocess_train_df: preprocess_train_df
- preprocess_test_df: preprocess_test_df
- split_sets: split_sets


# 1. INITIAL READING OF THE DATASET AND PREPROCESSING PIPELINES EXECUTION

In [17]:
# Reading CSV files with pandas

cancer_df = pd.read_csv('https://raw.githubusercontent.com/gascalero/DM_II_project/4a73ca4928f2b95f960cd9b9f44c4700244ed553/data/raw/patient_train_data.csv',
                        encoding='UTF-8',
                        index_col=0,
                        sep=',',
                        on_bad_lines='skip',
                        quoting=3)
cancer_df.head(1)

,Alcohol Consumption,Cancer Stage,Country,Date of Birth,Diabetes,Diabetes History,Diet Risk,Early Detection,Family History,Gender,...,Non Smoker,Obesity BMI,Physical Activity,Screening History,Smoking History,Transfusion History,Treatment Type,Tumor Size (mm),Urban or Rural,Survival Prediction
ID,,,,,,,,,,,,,,,,,,,,,
1,No,Localized,UK,29-01-1966,No,No,Moderate,No,No,M,...,Yes,Overweight,Low,Regular,No,-,Chemotherapy,33.0,Urban,Yes


In [18]:
# Splitting the dataset into training and validation sets
X_train, X_val, y_train, y_val = split_sets(cancer_df)

Dimension of X_train: (60028, 30)
Dimension of X_test: (15007, 30)
Dimension of y_train: (60028,)
Dimension of y_test: (15007,)


In [19]:
# Running the preprocessing pipeline for the training set
X_train_processed, y_train_processed, mode_train, stats_pre, encoders = preprocess_train_df(X_train, y_train, columns_to_delete, values_to_imput_cat, binary_cols, ordinal_mappings, numeric_cols, categorical_cols)

Dimension of X_clean after: (59180, 31)


In [22]:
# Running the preprocessing pipeline for the validation set

X_val_processed = preprocess_test_df(X_val, X_train_processed.columns, values_to_imput_cat, columns_to_delete, mode_train, stats_pre, binary_cols, numeric_cols, categorical_cols, encoders)

# y_val_processed = y_val.copy() for following the same convention as X_val_processed
y_val_processed = y_val.copy()

# 2. MODEL EVALUATION AND SELECTION (CROSS-VALIDATION)

In [23]:
# Define initial models with default or basic parameters
models = {
    'LogisticRegression': LogisticRegression(class_weight='balanced', max_iter=10000, random_state=42),
    'RandomForest': RandomForestClassifier(class_weight='balanced', random_state=42),
    'GradientBoosting': GradientBoostingClassifier(random_state=42),
    'KNeighbors': KNeighborsClassifier(weights='distance'),
    'XGBoost': XGBClassifier(eval_metric='aucpr', use_label_encoder=False, random_state=42)
}

This function is used for evaluating models using Cross-Validation. It returns the average measures after running the trained model on each fold's validation set and training set. This helps provide an idea of possible overfitting.

In [24]:
def evaluate_models_with_cv(X, y, models_dict, cv, scoring='f1'):
    cv_splitter = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)
    f1_macro_scorer = make_scorer(f1_score, average='weighted')
    
    results = []
    
    for name, model in models_dict.items():
        print(f"Evaluating {name}...")
        cv_scores = cross_val_score(
            model, X, y, 
            cv=cv_splitter,
            scoring=f1_macro_scorer,
            n_jobs=-1
        )
        
        # Calculate training scores for each fold
        train_scores = []
        for train_idx, val_idx in cv_splitter.split(X, y):
            X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
            y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]
            
            # Train the model on the training fold
            # Use clone to avoid fitting the original model
            model_clone = clone(model)
            model_clone.fit(X_train_fold, y_train_fold)
            
            # Evaluate on the training fold
            # Use the same model to predict on the training fold
            y_train_pred = model_clone.predict(X_train_fold)
            train_f1 = f1_score(y_train_fold, y_train_pred, average='weighted')
            train_scores.append(train_f1)
        
        # Calculate mean and std of training scores
        train_scores = np.array(train_scores)
        
        results.append({
            'Model': name,
            'Mean F1 Weighted (Val)': cv_scores.mean(),
            'Std F1 Weighted (Val)': cv_scores.std(),
            'Min F1 Weighted (Val)': cv_scores.min(),
            'Max F1 Weighted (Val)': cv_scores.max(),
            'Mean F1 Weighted (Train)': train_scores.mean(),
            'Std F1 Weighted (Train)': train_scores.std(),
            'Overfit Gap': train_scores.mean() - cv_scores.mean(),
            'Val Scores': cv_scores,
            'Train Scores': train_scores
        })
    
    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values('Mean F1 Weighted (Val)', ascending=False)
    
    return results_df

This function works the same way as the previous one, with the difference of applying the SMOTE technique for class balancing. This helps compare results and optimize model performance in case of better outcomes.

In [25]:
def evaluate_models_with_cv_smote(X, y, models_dict, cv, scoring='f1'):

    cv_splitter = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)
    f1_weighted_scorer = make_scorer(f1_score, average='weighted')
    
    results = []
    
    for name, model in models_dict.items():
        print(f"Evaluating {name}...")
        
        # We don't use cross_val_score here because we need to apply SMOTE in each fold manually
        cv_scores = []
        train_scores = []
        
        for train_idx, val_idx in cv_splitter.split(X, y):
            X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
            y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]
            
            # Apply SMOTE to the training fold
            smote = SMOTE(random_state=42)
            X_train_fold_balanced, y_train_fold_balanced = smote.fit_resample(X_train_fold, y_train_fold)
            
            # Train the model on the balanced training fold
            model_clone = clone(model)
            model_clone.fit(X_train_fold_balanced, y_train_fold_balanced)
            
            # Evaluate on the validation fold (not balanced)
            y_val_pred = model_clone.predict(X_val_fold)
            val_f1 = f1_score(y_val_fold, y_val_pred, average='weighted')
            cv_scores.append(val_f1)
            
            # Evaluate on the training fold (not balanced)
            y_train_pred = model_clone.predict(X_train_fold)
            train_f1 = f1_score(y_train_fold, y_train_pred, average='weighted')
            train_scores.append(train_f1)
        
        # Convert to numpy arrays for easier handling
        cv_scores = np.array(cv_scores)
        train_scores = np.array(train_scores)
        
        results.append({
            'Model': name,
            'Mean F1 Weighted (Val)': cv_scores.mean(),
            'Std F1 Weighted (Val)': cv_scores.std(),
            'Min F1 Weighted (Val)': cv_scores.min(),
            'Max F1 Weighted (Val)': cv_scores.max(),
            'Mean F1 Weighted (Train)': train_scores.mean(),
            'Std F1 Weighted (Train)': train_scores.std(),
            'Overfit Gap': train_scores.mean() - cv_scores.mean(),
            'Val Scores': cv_scores,
            'Train Scores': train_scores
        })
    
    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values('Mean F1 Weighted (Val)', ascending=False)
    
    return results_df

After the functions for cross-validation are designed, we execute them.

In [26]:
# Evaluate models with cross-validation without SMOTE
cv = 5 # Number of folds for cross-validation
cv_results = evaluate_models_with_cv(X_train_processed, y_train_processed, models, cv)

# Show results
cv_results[['Model', 'Mean F1 Weighted (Train)', 'Std F1 Weighted (Train)', 'Mean F1 Weighted (Val)', 'Std F1 Weighted (Val)' ,'Overfit Gap']]

Evaluating LogisticRegression...
Evaluating RandomForest...
Evaluating GradientBoosting...
Evaluating KNeighbors...
Evaluating XGBoost...


,Model,Mean F1 Weighted (Train),Std F1 Weighted (Train),Mean F1 Weighted (Val),Std F1 Weighted (Val),Overfit Gap
3,KNeighbors,1.000000,0.000000,0.529040,0.002972,0.470960
4,XGBoost,0.713233,0.003493,0.511637,0.002096,0.201596
0,LogisticRegression,0.521614,0.000579,0.511357,0.003584,0.010257
1,RandomForest,1.000000,0.000000,0.481316,0.002784,0.518684
2,GradientBoosting,0.456086,0.001162,0.451857,0.000511,0.004230


In [27]:
# Evaluate models with cross-validation and SMOTE
cv = 5 # Number of folds for cross-validation
cv_results = evaluate_models_with_cv_smote(X_train_processed, y_train_processed, models, cv)

# Show results
cv_results[['Model', 'Mean F1 Weighted (Train)', 'Std F1 Weighted (Train)', 'Mean F1 Weighted (Val)', 'Std F1 Weighted (Val)' ,'Overfit Gap']]

Evaluating LogisticRegression...
Evaluating RandomForest...
Evaluating GradientBoosting...
Evaluating KNeighbors...
Evaluating XGBoost...


,Model,Mean F1 Weighted (Train),Std F1 Weighted (Train),Mean F1 Weighted (Val),Std F1 Weighted (Val),Overfit Gap
0,LogisticRegression,0.521296,0.000831,0.517634,0.005089,0.003662
3,KNeighbors,1.000000,0.000000,0.511884,0.004556,0.488116
1,RandomForest,1.000000,0.000000,0.511232,0.001338,0.488768
4,XGBoost,0.686901,0.004057,0.509011,0.002306,0.177890
2,GradientBoosting,0.452348,0.001397,0.450318,0.000099,0.002031


# 3. HYPERPARAMETERS OPTIMIZATION

After the candidate models are selected, we perform a GridSearch for tuning hyperparameters to maximize performance results.

## 3.1. LOGISTIC REGRESSION

In [30]:
param_grid = {
    'classifier__C': [0.01, 0.1, 1, 10, 20, 50, 100],
    'classifier__penalty': ['l2'],
    'classifier__solver': ['liblinear', 'lbfgs', 'saga'],
    'classifier__class_weight': ['balanced'],
    'classifier__max_iter': [1000000]
}

# Create a pipeline with SMOTE and Logistic Regression. We use SMOTE to balance the classes in the training set.
pipeline = Pipeline([
    ('smote', SMOTE(random_state=42, k_neighbors=5, sampling_strategy=0.8)),
    ('classifier', LogisticRegression(random_state=42))
])

# Define the scoring function
scorer = make_scorer(f1_score, average='weighted')

# Configure GridSearchCV
# Use the pipeline and the parameter grid for Logistic Regression
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring=scorer,
    cv=5,  # Validación cruzada de 5 folds
    n_jobs=-1,  # Usar todos los núcleos disponibles
    verbose=0,   # Reducir la verbosidad para evitar problemas en Jupyter Notebook
    return_train_score=True
)

# Fit the model with the training data
grid_search.fit(X_train_processed, y_train_processed)

# Show best parameters and scores
print("Best parameters:")
print(grid_search.best_params_)
print(f"Best F1 score: {grid_search.best_score_:.4f}")

# Create model with the best parameters
best_params = {key.replace('classifier__', ''): value for key, value in grid_search.best_params_.items()}
best_log_reg = LogisticRegression(**best_params, random_state=42)

# Train the model with the best parameters on the entire training set
best_log_reg.fit(X_train_processed, y_train_processed)

# Evaluate the model on the validation set
y_pred_best = best_log_reg.predict(X_val_processed)
print("\nClassification report with best parameters:")
print(classification_report(y_val_processed, y_pred_best))

Best parameters:
{'classifier__C': 0.01, 'classifier__class_weight': 'balanced', 'classifier__max_iter': 1000000, 'classifier__penalty': 'l2', 'classifier__solver': 'saga'}
Best F1 score: 0.5093

Classification report with best parameters:
              precision    recall  f1-score   support

           0       0.40      0.48      0.43      5976
           1       0.60      0.52      0.56      9031

    accuracy                           0.50     15007
   macro avg       0.50      0.50      0.49     15007
weighted avg       0.52      0.50      0.51     15007



# 4. KAGGLE SUBMISSION CREATION

After selecting the best model with its optimal hyperparameters, it is now time to create the final CSV for submission in the Kaggle competition.

In [ ]:
# Importing the test dataset provided in the DM II project materials

test_cancer_df = pd.read_csv('https://raw.githubusercontent.com/gascalero/DM_II_project/refs/heads/master/data/raw/patient_test_data.csv',
                         encoding='UTF-8',
                         index_col=0,
                         sep=',',
                         on_bad_lines='skip',
                         quoting=3)

test_cancer_df.head(1)

In [ ]:
# Executing the preprocessing pipeline for the final test set

X_test_processed = preprocess_test_df(test_cancer_df, X_train_processed.columns, values_to_imput_cat, columns_to_delete, mode_train, stats_pre, binary_cols, numeric_cols, categorical_cols, encoders)

In [ ]:
# Running the selected model into the test set

logistic_model = LogisticRegression(C= 20, class_weight= 'balanced', max_iter= 1000000, penalty= 'l2', solver= 'lbfgs')

X_train_processed_balanced, y_train_processed_balanced = SMOTE(random_state=42).fit_resample(X_train_processed, y_train_processed)
logistic_model.fit(X_train_processed_balanced, y_train_processed_balanced)
y_pred_logistic = logistic_model.predict(X_test_processed)

In [ ]:
# Creating the DataFrame for Kaggle submission
df_kaggle = pd.DataFrame(y_pred_logistic, index=test_cancer_df.index)
df_kaggle.replace({0: 'No', 1: 'Yes'}, inplace = True)
df_kaggle.columns = ['Survival Prediction']
df_kaggle.value_counts()

In [ ]:
# Exporting the DataFrame to a CSV file for submission
# Modify the file name as needed with the corresponding version number
df_kaggle.to_csv('/data/Kaggle submissions/DT_Group05_VersionXX.csv')